# [ALL-IN-ONE PIPELINE] HUẤN LUYỆN TOÀN DIỆN 4 MÔ HÌNH TRÊN BIGCLONEBENCH JAVA (GPU)

Notebook này tự động huấn luyện và đánh giá liên tục cả **4 phương pháp phát hiện clone ngữ nghĩa**:
1. **M3 (ASTNN):** Tree-based Neural Network trên cấu trúc Abstract Syntax Tree.
2. **M1 (CodeBERT):** Sequence-based Transformer (Pretrained CodeBERT base).
3. **M2 (GraphCodeBERT):** Pretrained Transformer tích hợp Data Flow Graph.
4. **M4 (CPG-GNN):** Graph Neural Network (GAT) trên Code Property Graph xuất từ Joern.

### Thông tin cấu hình:
- **Dataset:** `semanticcodeclonedetection-dataset-m4` (đã bao gồm AST Cache và CPG Graphs).
- **Phân tập huấn luyện:** Train 50.000 cặp, Validation 5.000 cặp, Test 5.000 cặp BigCloneBench.
- **Quản lý lỗi độc lập:** Mỗi mô hình chạy trong khối try/finally độc lập, giải phóng GPU VRAM và lưu kết quả ngay khi xong.
- **Kết quả cuối cùng:** Toàn bộ checkpoint, metric chi tiết từng loại clone (Type-1 -> Type-4) và file nén `bcb_all_models_results.tar.gz` được lưu tại `/kaggle/working/`.

---

In [ ]:
# 1. THIẾT LẬP MÔI TRƯỜNG & HÀM BỔ TRỢ
import base64, io, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

def sh(cmd, **kw):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.stdout:
        print(r.stdout.strip())
    if r.stderr:
        print("[stderr]", r.stderr.strip())
    if r.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (rc={r.returncode}): {cmd}")
    return r.stdout

print("Python:", sys.version.split()[0])

In [ ]:
# 2. GIẢI NÉN MÃ NGUỒN DỰ ÁN MỚI NHẤT (SELF-CONTAINED)
WORK_DIR = Path("/kaggle/working/project")
WORK_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_B64 = "..."

print(f"Giải nén mã nguồn ({len(BUNDLE_B64)/1024:.1f} KB) vào {WORK_DIR}...")
buf = io.BytesIO(base64.b64decode(BUNDLE_B64))
with tarfile.open(fileobj=buf, mode="r:gz") as tar:
    tar.extractall(WORK_DIR)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
sh("ls -la configs/")

In [ ]:
# 3. CÀI ĐẶT THƯ VIỆN & XÁC THỰC GPU
sh("pip install -q 'tree-sitter>=0.23.0' 'tree-sitter-java>=0.23.5' 'tree-sitter-cpp>=0.23.0' 'datasets>=2.14.0' 'transformers>=4.30.0' 'pyarrow' 'tqdm' 'pyyaml' 'scikit-learn'")

import torch
assert torch.cuda.is_available(), "Cần bật GPU trong Settings -> Accelerator -> GPU (T4/P100)!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print(f"GPU SẴN SÀNG: {gpu_name} | VRAM: {vram_gb} GB")
sh("nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv")

In [ ]:
# 4. LIÊN KẾT ĐẶC TRƯNG TỪ DATASET (AST CACHE & CPG EXPORT)
in_dir = Path("/kaggle/input/semanticcodeclonedetection-dataset-m4")
if in_dir.exists():
    print(f"Dataset đính kèm: {in_dir}")
    sh(f"ls -lh {in_dir}")
    # Nạp AST cache cho M3
    ast_p = in_dir / "bcb_ast_cache.npz"
    if ast_p.exists():
        os.makedirs("artifacts/bcb_features", exist_ok=True)
        sh(f"cp {ast_p} artifacts/bcb_features/")
        print("Đã nạp bcb_ast_cache.npz thành công.")
    # Nạp CPG archive cho M4
    cpg_tar = in_dir / "bcb_cpg_export.tar.gz"
    if cpg_tar.exists():
        os.makedirs("artifacts", exist_ok=True)
        print("Đang giải nén bcb_cpg_export.tar.gz...")
        sh(f"tar -xzf {cpg_tar} -C artifacts/")
        cpg_count = len(list(Path("artifacts/cpg_export").glob("*.json")))
        print(f"Đã giải nén {cpg_count:,} đồ thị CPG JSON vào artifacts/cpg_export/")
else:
    print("CẢNH BÁO: Không tìm thấy thư mục dataset đính kèm!")

In [ ]:
# 5. TẢI CÁC PHÂN TẬP DỮ LIỆU ĐỂ HUẤN LUYỆN (TRAIN 50K, VAL 5K, TEST 5K)
data_raw = Path("data/raw/bigclonebench")
data_raw.mkdir(parents=True, exist_ok=True)
print("Tải các phân tập dữ liệu BigCloneBench (Train: 50k, Val: 5k, Test: 5k) qua streaming...")
sh("python scripts/download_bcb_subsets.py --split train --max-samples 50000")
sh("python scripts/download_bcb_subsets.py --split validation --max-samples 5000")
sh("python scripts/download_bcb_subsets.py --split test --max-samples 5000")
sh("ls -lh data/raw/bigclonebench/")

In [ ]:
# 6. HUẤN LUYỆN & ĐÁNH GIÁ M3 (ASTNN)
print("=" * 70)
print(">>> [1/4] BẮT ĐẦU HUẤN LUYỆN M3 (ASTNN) - TREE-BASED NEURAL NETWORK...")
print("=" * 70)
try:
    sh("python -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode preflight --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode diagnostic --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode train --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode evaluate --device cuda")
    m3_res = Path("results/exp_bcb_004")
    m3_out = Path("/kaggle/working/m3_results")
    m3_out.mkdir(parents=True, exist_ok=True)
    if m3_res.exists():
        sh(f"cp -r {m3_res}/* {m3_out}/")
        sh(f"cp {m3_out}/test_metrics.json /kaggle/working/m3_test_metrics.json 2>/dev/null || true")
    print(">>> [1/4] M3 (ASTNN) HOÀN TẤT THÀNH CÔNG!")
except Exception as e:
    print(f">>> [1/4] LỖI TẠI M3 (ASTNN): {e}")
finally:
    import torch; torch.cuda.empty_cache()
    print("Đã giải phóng GPU VRAM sau khi chạy M3.")

In [ ]:
# 7. HUẤN LUYỆN & ĐÁNH GIÁ M1 (CODEBERT)
print("=" * 70)
print(">>> [2/4] BẮT ĐẦU HUẤN LUYỆN M1 (CODEBERT) - SEQUENCE TRANSFORMER...")
print("=" * 70)
try:
    sh("python -m src.experiments.bcb_runner --config configs/exp002_codebert_bcb.yaml --mode preflight --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp002_codebert_bcb.yaml --mode diagnostic --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp002_codebert_bcb.yaml --mode train --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp002_codebert_bcb.yaml --mode evaluate --device cuda")
    m1_res = Path("results/exp_bcb_002")
    m1_out = Path("/kaggle/working/m1_results")
    m1_out.mkdir(parents=True, exist_ok=True)
    if m1_res.exists():
        sh(f"cp -r {m1_res}/* {m1_out}/")
        sh(f"cp {m1_out}/test_metrics.json /kaggle/working/m1_test_metrics.json 2>/dev/null || true")
    print(">>> [2/4] M1 (CODEBERT) HOÀN TẤT THÀNH CÔNG!")
except Exception as e:
    print(f">>> [2/4] LỖI TẠI M1 (CODEBERT): {e}")
finally:
    import torch; torch.cuda.empty_cache()
    print("Đã giải phóng GPU VRAM sau khi chạy M1.")

In [ ]:
# 8. HUẤN LUYỆN & ĐÁNH GIÁ M2 (GRAPHCODEBERT)
print("=" * 70)
print(">>> [3/4] BẮT ĐẦU HUẤN LUYỆN M2 (GRAPHCODEBERT) - DFG TRANSFORMER...")
print("=" * 70)
try:
    sh("python -m src.experiments.bcb_runner --config configs/exp003_graphcodebert_bcb.yaml --mode preflight --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp003_graphcodebert_bcb.yaml --mode diagnostic --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp003_graphcodebert_bcb.yaml --mode train --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp003_graphcodebert_bcb.yaml --mode evaluate --device cuda")
    m2_res = Path("results/exp_bcb_003")
    m2_out = Path("/kaggle/working/m2_results")
    m2_out.mkdir(parents=True, exist_ok=True)
    if m2_res.exists():
        sh(f"cp -r {m2_res}/* {m2_out}/")
        sh(f"cp {m2_out}/test_metrics.json /kaggle/working/m2_test_metrics.json 2>/dev/null || true")
    print(">>> [3/4] M2 (GRAPHCODEBERT) HOÀN TẤT THÀNH CÔNG!")
except Exception as e:
    print(f">>> [3/4] LỖI TẠI M2 (GRAPHCODEBERT): {e}")
finally:
    import torch; torch.cuda.empty_cache()
    print("Đã giải phóng GPU VRAM sau khi chạy M2.")

In [ ]:
# 9. HUẤN LUYỆN & ĐÁNH GIÁ M4 (CPG-GNN)
print("=" * 70)
print(">>> [4/4] BẮT ĐẦU HUẤN LUYỆN M4 (CPG-GNN) - GRAPH NEURAL NETWORK ON JOERN CPG...")
print("=" * 70)
try:
    sh("python -m src.experiments.bcb_runner --config configs/exp005_cpg_gnn_bcb.yaml --mode preflight --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp005_cpg_gnn_bcb.yaml --mode diagnostic --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp005_cpg_gnn_bcb.yaml --mode train --device cuda")
    sh("python -m src.experiments.bcb_runner --config configs/exp005_cpg_gnn_bcb.yaml --mode evaluate --device cuda")
    m4_res = Path("results/exp_bcb_005")
    m4_out = Path("/kaggle/working/m4_results")
    m4_out.mkdir(parents=True, exist_ok=True)
    if m4_res.exists():
        sh(f"cp -r {m4_res}/* {m4_out}/")
        sh(f"cp {m4_out}/test_metrics.json /kaggle/working/m4_test_metrics.json 2>/dev/null || true")
    print(">>> [4/4] M4 (CPG-GNN) HOÀN TẤT THÀNH CÔNG!")
except Exception as e:
    print(f">>> [4/4] LỖI TẠI M4 (CPG-GNN): {e}")
finally:
    import torch; torch.cuda.empty_cache()
    print("Đã giải phóng GPU VRAM sau khi chạy M4.")

In [ ]:
# 10. TỔNG HỢP SO SÁNH TOÀN BỘ CÁC MÔ HÌNH & ĐÓNG GÓI KẾT QUẢ
print("=" * 70)
print("TỔNG HỢP BÁO CÁO TOÀN DIỆN VÀ ĐÓNG GÓI KẾT QUẢ...")
print("=" * 70)

models_info = [
    ("M3", "ASTNN", "/kaggle/working/m3_results/test_metrics.json"),
    ("M1", "CodeBERT", "/kaggle/working/m1_results/test_metrics.json"),
    ("M2", "GraphCodeBERT", "/kaggle/working/m2_results/test_metrics.json"),
    ("M4", "CPG-GNN", "/kaggle/working/m4_results/test_metrics.json"),
]

comparison = {}
print("\n" + "-" * 80)
print(f"{'Model':<6} | {'Name':<15} | {'F1-Score':<10} | {'Precision':<10} | {'Recall':<10} | {'Accuracy':<10}")
print("-" * 80)

for mid, mname, mp in models_info:
    p = Path(mp)
    if p.exists():
        data = json.loads(p.read_text(encoding='utf-8'))
        comparison[mid] = data
        f1 = data.get('f1', 0.0)
        prec = data.get('precision', 0.0)
        rec = data.get('recall', 0.0)
        acc = data.get('accuracy', 0.0)
        print(f"{mid:<6} | {mname:<15} | {f1:<10.4f} | {prec:<10.4f} | {rec:<10.4f} | {acc:<10.4f}")
    else:
        print(f"{mid:<6} | {mname:<15} | {'CHƯA CÓ':<10} | {'-':<10} | {'-':<10} | {'-':<10}")
print("-" * 80)

# Lưu file JSON tổng hợp
summary_path = Path("/kaggle/working/all_models_summary.json")
summary_path.write_text(json.dumps(comparison, indent=2, ensure_ascii=False), encoding='utf-8')

# Nén toàn bộ kết quả
archive_path = Path("/kaggle/working/bcb_all_models_results.tar.gz")
print(f"Đang nén toàn bộ kết quả vào {archive_path}...")
sh(f"tar -czf {archive_path} results/ /kaggle/working/*_results /kaggle/working/*.json 2>/dev/null || tar -czf {archive_path} results/")
sh("ls -lh /kaggle/working/")

print("\n======================================================================")
print("HOÀN TẤT TOÀN BỘ TIẾN TRÌNH HUẤN LUYỆN 4 MÔ HÌNH TRÊN BIGCLONEBENCH!")
print("======================================================================")